# 실험 03: RAG 실행 과정 전체 파이프라인 (검색 + 생성)

## 1. 실험 제목과 목표
- **제목**: 오픈북 시험 치르는 AI 만들기
- **목표**: 교안 6번과 9번 슬라이드의 전체 과정(`질문 -> 검색 -> 근거 추가 -> 답변`)을 하나로 엮는 완전한 RAG 체인을 구축합니다.

## 2. 실험 개요
1. **실험 1 (일반 LLM)**: 모델이 학습하지 않은 사내 정보에 대해 물어봤을 때 어떻게 대답하는지(모른다고 하거나 헛소리를 함) 확인합니다.
2. **실험 2 (RAG 체인)**: LangChain의 `create_retrieval_chain`을 이용해 문서를 찾아 프롬프트에 끼워 넣고 정답을 말하게 만듭니다.
3. **실패/주의 케이스**: Retriever가 찾아온 문서에 정답이 없을 때 발생하는 환각(Hallucination) 현상을 관찰합니다.

## 3. 라이브러리 import 및 이전 과정(Vector DB) 세팅
역시 이전 과정의 코드를 가져와서 메모리 DB를 띄웁니다.

In [2]:
import os
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

load_dotenv()
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini")

docs = [
    Document(page_content="[제2장 복지] 재택근무는 주 2회 가능하며, 팀장의 사전 승인이 필수입니다."),
    Document(page_content="[제3장 휴가] 생일자에게는 오후 반차를 부여합니다.")
]
vectorstore = Chroma.from_documents(documents=docs, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

## 4. 실험 1: 일반 LLM의 한계 (지식의 부재)

In [3]:
print("=== [RAG 없는 일반 LLM의 답변] ===")
question = "우리 회사 재택근무 규정 알려줘."

normal_res = llm.invoke(question)
print("질문:", question)
print("답변:", normal_res.content)
print("\n-> 결과: OpenAI는 우리 회사의 규정을 학습한 적이 없으므로, 일반론적인 이야기(각 회사마다 다릅니다 등)를 하거나 거짓말을 할 수밖에 없습니다.")

=== [RAG 없는 일반 LLM의 답변] ===
질문: 우리 회사 재택근무 규정 알려줘.
답변: 재택근무 규정은 각 회사의 정책과 업무 환경에 따라 다를 수 있지만, 일반적으로 포함되는 주요 내용은 다음과 같습니다:

1. **근무 시간**: 재택근무 시 근무 시간은 정해진 출퇴근 시간을 준수해야 하며, 유연근무제를 적용할 경우 사전에 조율해야 합니다.

2. **업무 보고**: 정기적인 업무 보고 및 회의 참여를 의무화하여 팀과의 소통을 유지합니다. 주간/월간 보고서 제출이 요구될 수 있습니다.

3. **보안 규정**: 회사의 기밀 정보를 보호하기 위해 안전한 네트워크를 사용하고, 개인 기기를 사용 시 보안 소프트웨어 설치가 필요합니다.

4. **업무 장비**: 필요한 장비(컴퓨터, 모니터 등)는 회사에서 제공하거나, 개인 장비 사용 시 사전 승인 받도록 규정할 수 있습니다.

5. **생산성 유지**: 재택근무 중 생산성을 유지하기 위해 정기적인 업무 계획 수립과 자기 관리가 요구됩니다.

6. **기타**: 재택근무 중 건강 및 복지를 위한 시간 관리, 정해진 휴식 시간 준수 등의 내용도 포함될 수 있습니다.

자세한 내용은 회사의 인사팀이나 내부 규정을 참조하는 것이 좋습니다.

-> 결과: OpenAI는 우리 회사의 규정을 학습한 적이 없으므로, 일반론적인 이야기(각 회사마다 다릅니다 등)를 하거나 거짓말을 할 수밖에 없습니다.


## 5. 실험 2: RAG 파이프라인 (검색 -> 생성 체인)

In [4]:
print("=== [오픈북 시험: RAG 답변] ===")

# 1. 프롬프트 템플릿 작성 (검색된 문서를 넣을 {context} 구멍을 뚫어놓습니다)
prompt = ChatPromptTemplate.from_template("""
다음 제공된 문서를 바탕으로 사용자의 질문에 답하세요.

[문서]
{context}

[질문]
{input}
""")

# 2. 문서 결합 체인 생성 (검색된 여러 개의 문서를 하나의 긴 문자열로 합쳐서 {context}에 쑤셔넣는 역할)
document_chain = create_stuff_documents_chain(llm, prompt)

# 3. 최종 RAG 체인 조립 (Retriever + 문서 결합 체인)
rag_chain = create_retrieval_chain(retriever, document_chain)

# 실행!
res = rag_chain.invoke({"input": question})

print("질문:", question)
print("답변:", res["answer"])
print("\n-> 결과: 사내 문서를 먼저 검색하고, 프롬프트의 {context} 자리에 몰래 끼워넣은 뒤 LLM에게 보여주었기 때문에, 완벽하게 사내 규정을 설명해 냅니다.")

=== [오픈북 시험: RAG 답변] ===
질문: 우리 회사 재택근무 규정 알려줘.
답변: 우리 회사의 재택근무 규정에 따르면, 재택근무는 주 2회 가능하며, 팀장의 사전 승인이 필요합니다.

-> 결과: 사내 문서를 먼저 검색하고, 프롬프트의 {context} 자리에 몰래 끼워넣은 뒤 LLM에게 보여주었기 때문에, 완벽하게 사내 규정을 설명해 냅니다.


## 6. 실패/주의 케이스: 문서에 정답이 없을 때 발생하는 환각
우리가 질문을 던졌지만, Retriever가 정답이 없는 쓸데없는 문서를 가져왔을 경우 모델은 어떻게 반응할까요?

In [5]:
print("[정답이 없는 문서 상황 시뮬레이션]")
bad_question = "회식비 지원은 얼마나 돼?"

bad_res = rag_chain.invoke({"input": bad_question})

print("질문:", bad_question)
# 디버깅: Retriever가 대체 무슨 문서를 가져왔는지 확인해봅시다
print("[참고한 문서 (Retriever가 엉뚱하게 가져옴)]")
for doc in bad_res["context"]:
    print("->", doc.page_content)

print("\n[최종 답변]")
print(bad_res["answer"])

print("\n-> 주의: DB에 회식비 내용이 없으니 Retriever는 그나마 제일 비슷해 보이는(사실은 전혀 관련 없는) 복지/휴가 문서를 가져옵니다.")
print("   그런데 모델이 '회식비 지원에 대한 정보는 제공된 문서에 없습니다'라고 솔직히 말하지 않고, '회식은 주 2회 가능합니다(재택근무 규정을 꼬아서)' 라고 지어낼 위험이 큽니다.")

[정답이 없는 문서 상황 시뮬레이션]
질문: 회식비 지원은 얼마나 돼?
[참고한 문서 (Retriever가 엉뚱하게 가져옴)]
-> [제3장 휴가] 생일자에게는 오후 반차를 부여합니다.

[최종 답변]
문서에는 회식비 지원에 대한 내용이 포함되어 있지 않습니다. 따라서 회식비 지원에 대한 구체적인 정보를 제공할 수 없습니다. 추가적인 정보를 원하시면 관련 부서나 정책을 확인해 보시기 바랍니다.

-> 주의: DB에 회식비 내용이 없으니 Retriever는 그나마 제일 비슷해 보이는(사실은 전혀 관련 없는) 복지/휴가 문서를 가져옵니다.
   그런데 모델이 '회식비 지원에 대한 정보는 제공된 문서에 없습니다'라고 솔직히 말하지 않고, '회식은 주 2회 가능합니다(재택근무 규정을 꼬아서)' 라고 지어낼 위험이 큽니다.


## 7. 결과 해석

1. **`create_retrieval_chain`**: 개발자가 직접 검색하고, 텍스트를 합치고, 프롬프트에 넣고 LLM을 호출하는 수고를 덜어주는 LangChain의 강력한 유틸리티입니다.
2. **환각(Hallucination)**: RAG를 구축한다고 해서 환각이 완벽히 사라지는 것이 아닙니다. 엉뚱한 문서를 줬을 때 LLM이 무리해서 답을 지어내려는 성향을 막는 장치가 필요합니다.

## 8. Notion 실험 로그

### 발견한 문제점 (또는 차이점)
* 일반 LLM은 모르는 회사 내부 지식도 RAG를 태우면 완벽하게 답변함.
* 딕셔너리로 반환되며, `res["answer"]`에는 최종 텍스트가, `res["context"]`에는 참고했던 원본 문서 리스트가 고스란히 담겨 나옴.
* 하지만 관련 없는 문서를 줘버렸을 때 거짓말을 치는 치명적인 약점을 발견함.

### 다음 개선 방향
* 교안 10번 슬라이드에 나온 **고급 RAG 기법(Grounding)**을 도입해서, 모르면 모른다고 당당하게 말하게 만들고, 답변에 **출처(Citation)**를 남기도록 프롬프트를 튜닝하는 4번째 실전 노트북으로 넘어갈 차례임.